# Notebook 06: Fay-Herriot Small Area Estimation

Applies Fay-Herriot shrinkage to reduce variance of IPW estimates by borrowing strength from a gravity model.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path
import h3
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

CRS_GEO = "EPSG:4674"
CRS_UTM = "EPSG:32719"

## Load data

In [ ]:
od = pd.read_parquet(DATA / "od_comuna_ipw.parquet")
print(f"Comuna OD pairs: {len(od):,}")

detection = pd.read_parquet(DATA / "detection_floor.parquet")
h3_pop = pd.read_parquet(DATA / "h3_population.parquet")

comunas = gpd.read_file(CENSUS, layer="Comunal_CPV24")
comunas["CUT"] = comunas["CUT"].astype(int)
comunas_utm = comunas.to_crs(CRS_UTM)

comunas["centroid"] = comunas_utm.geometry.centroid
comunas["cx"] = comunas["centroid"].x
comunas["cy"] = comunas["centroid"].y

print(f"Comunas: {len(comunas)}")

## Comuna-level covariates

In [ ]:
bts = pd.read_parquet(DATA / "bts_sites.parquet")
bts["h3_res8"] = [h3.latlng_to_cell(lat, lon, 8) for lat, lon in zip(bts["lat"], bts["lon"])]

bts_geo = gpd.GeoDataFrame(
    bts, geometry=gpd.points_from_xy(bts.lon, bts.lat), crs=CRS_GEO,
)
bts_comuna = gpd.sjoin(bts_geo, comunas[["CUT", "geometry"]], how="left", predicate="within")

comuna_area = comunas_utm.copy()
comuna_area["area_km2"] = comuna_area.geometry.area / 1e6

n_towers = bts_comuna.groupby("CUT").size().rename("n_towers")
comuna_stats = comuna_area[["CUT", "area_km2"]].merge(n_towers, on="CUT", how="left")
comuna_stats["n_towers"] = comuna_stats["n_towers"].fillna(0)
comuna_stats["tower_density"] = comuna_stats["n_towers"] / comuna_stats["area_km2"]

mean_reff = (
    bts_comuna.merge(detection[["bts_id", "effective_resolution"]],
                      left_on="bts_id", right_on="bts_id", how="left")
    .groupby("CUT")["effective_resolution"]
    .mean()
    .rename("mean_r_eff")
)
comuna_stats = comuna_stats.merge(mean_reff, on="CUT", how="left")
comuna_stats["mean_r_eff"] = comuna_stats["mean_r_eff"].fillna(
    comuna_stats["mean_r_eff"].max()
)

h3_pts = gpd.GeoDataFrame(
    h3_pop, geometry=gpd.points_from_xy(h3_pop["lon"], h3_pop["lat"]), crs=CRS_GEO,
)
h3_comuna = gpd.sjoin(h3_pts, comunas[["CUT", "geometry"]], how="left", predicate="within")
pop_by_comuna = h3_comuna.groupby("CUT")["pop_sector"].sum().rename("population")

comuna_stats = comuna_stats.merge(pop_by_comuna, on="CUT", how="left")
comuna_stats["population"] = comuna_stats["population"].fillna(0)

print("Comuna stats:")
print(comuna_stats[["CUT", "area_km2", "n_towers", "tower_density",
                     "mean_r_eff", "population"]].describe().to_string())

## Prepare Fay-Herriot inputs

In [ ]:
od = od.merge(
    comuna_stats[["CUT", "population", "tower_density", "mean_r_eff", "area_km2"]]
    .rename(columns={"CUT": "codcom_from", "population": "pop_from",
                      "tower_density": "td_from", "mean_r_eff": "reff_from",
                      "area_km2": "area_from"}),
    on="codcom_from", how="left",
)
od = od.merge(
    comuna_stats[["CUT", "population", "tower_density", "mean_r_eff", "area_km2"]]
    .rename(columns={"CUT": "codcom_to", "population": "pop_to",
                      "tower_density": "td_to", "mean_r_eff": "reff_to",
                      "area_km2": "area_to"}),
    on="codcom_to", how="left",
)

cx = comunas.set_index("CUT")[["cx", "cy"]]
od = od.merge(cx.rename(columns={"cx": "cx_from", "cy": "cy_from"}),
              left_on="codcom_from", right_index=True, how="left")
od = od.merge(cx.rename(columns={"cx": "cx_to", "cy": "cy_to"}),
              left_on="codcom_to", right_index=True, how="left")
od["distance_m"] = np.hypot(od["cx_from"] - od["cx_to"], od["cy_from"] - od["cy_to"])

od["log_pop_from"] = np.log1p(od["pop_from"])
od["log_pop_to"]   = np.log1p(od["pop_to"])
od["log_distance"] = np.log1p(od["distance_m"])
od["is_self_loop"] = (od["codcom_from"] == od["codcom_to"]).astype(float)

od["psi"] = (od["reff_from"] + od["reff_to"]) / 2
od["psi"] = od["psi"].fillna(od["psi"].max())

od["y"] = np.log1p(od["ipw_trips"])

od_fh = od[od["ipw_trips"] > 0].copy()
print(f"OD pairs for Fay-Herriot: {len(od_fh):,}")

## OLS structural model (gravity specification)

In [ ]:
X_cols = ["log_pop_from", "log_pop_to", "log_distance", "is_self_loop"]
X = sm.add_constant(od_fh[X_cols])
y = od_fh["y"]

ols = sm.OLS(y, X).fit()
print(ols.summary().tables[1])

od_fh["y_hat"] = ols.predict(X)
od_fh["residual"] = od_fh["y"] - od_fh["y_hat"]

## Model variance and shrinkage weights

In [ ]:
od_fh["psi_log"] = od_fh["psi"] / od_fh["ipw_trips"].clip(lower=1)

residual_var = od_fh["residual"].var()
mean_psi_log = od_fh["psi_log"].mean()
sigma_u2 = max(residual_var - mean_psi_log, 0.01)

print(f"Residual variance:   {residual_var:.4f}")
print(f"Mean sampling var:   {mean_psi_log:.4f}")
print(f"Model variance sigma_u2: {sigma_u2:.4f}")

od_fh["gamma"] = sigma_u2 / (sigma_u2 + od_fh["psi_log"])

print(f"\nShrinkage weight (gamma):")
print(od_fh["gamma"].describe().to_string())

## Fay-Herriot composite estimate

In [ ]:
od_fh["y_fh"] = od_fh["gamma"] * od_fh["y"] + (1 - od_fh["gamma"]) * od_fh["y_hat"]
od_fh["fh_trips"] = np.expm1(od_fh["y_fh"])

print(f"Total raw trips:    {od_fh['raw_trips'].sum():,.0f}")
print(f"Total IPW trips:    {od_fh['ipw_trips'].sum():,.0f}")
print(f"Total FH trips:     {od_fh['fh_trips'].sum():,.0f}")

## Figure -- Shrinkage map (Fig. 7 in paper)

In [ ]:
mean_gamma = od_fh.groupby("codcom_from")["gamma"].mean().rename("mean_gamma")
comunas_gamma = comunas.merge(mean_gamma, left_on="CUT", right_index=True, how="left")

fig, ax = plt.subplots(figsize=(10, 12))
comunas.boundary.plot(ax=ax, color="#cccccc", linewidth=0.3)
comunas_gamma.dropna(subset=["mean_gamma"]).plot(
    ax=ax, column="mean_gamma", cmap="RdYlGn",
    alpha=0.7, edgecolor="grey", linewidth=0.3,
    legend=True, legend_kwds={"label": "Mean shrinkage weight $\\gamma$", "shrink": 0.5},
    vmin=0, vmax=1,
)
ax.set_title("Fay-Herriot shrinkage weight per origin comuna\n"
             "($\\gamma \\to 1$: trust data; $\\gamma \\to 0$: trust model)")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(FIGURES / "fig_fh_shrinkage_map.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Three-way comparison for example comunas

In [ ]:
examples = {
    "Santiago (13101)": 13101,
    "Maipu (13119)": 13119,
    "Pirque (13103)": 13103,
}

for label, cut in examples.items():
    sub = od_fh[od_fh["codcom_from"] == cut].copy()
    if len(sub) == 0:
        continue
    print(f"\n{label}:")
    print(f"  OD pairs: {len(sub)}")
    print(f"  Raw total:  {sub['raw_trips'].sum():,.0f}")
    print(f"  IPW total:  {sub['ipw_trips'].sum():,.0f}")
    print(f"  FH total:   {sub['fh_trips'].sum():,.0f}")
    print(f"  Mean gamma: {sub['gamma'].mean():.3f}")

## Save outputs

In [ ]:
od_save = od_fh[["codcom_from", "codcom_to", "raw_trips", "ipw_trips",
                  "fh_trips", "gamma", "psi", "distance_m",
                  "pop_from", "pop_to"]].copy()
od_save.to_parquet(DATA / "od_comuna_fh.parquet", index=False)
print(f"Saved: od_comuna_fh.parquet ({len(od_save):,} rows)")